In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [27]:
df = pd.read_csv("../data/raw/venta_idealista.csv",on_bad_lines="warn")
df.head()


C:\Users\PDE6221\AppData\Local\Temp\ipykernel_25712\2889665582.py:1: ParserWarning: Skipping line 38: expected 22 fields, saw 23

  df = pd.read_csv("../data/raw/venta_idealista.csv",on_bad_lines="warn")


,id_anuncio,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,...,descuento_pct,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo
1,Inmobiliaria E-Castro,venta,piso,NaN,"""Avenida Riomar","1""",Cotolino,Castro-Urdiales,Cantabria,498000,...,NaN,sí,4,121,4,sí,sí,otros,0,NaN
2,Inmobiliarias.com,venta,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000,NaN,...,sí,5,232,NaN,desconocido,desconocido,otros,0,NaN,NaN
3,Inmobiliaria E-Castro,venta,piso,NaN,"""Calle Leonardo Rucabado","6""",Centro,Castro-Urdiales,Cantabria,299000,...,NaN,desconocido,2,84,3,sí,sí,otros,0,Garaje no indicado
4,Inmobiliaria E-Castro,venta,piso,NaN,"""Calle de la Ronda","32""",Centro,Castro-Urdiales,Cantabria,340000,...,NaN,desconocido,4,99,5,sí,sí,otros,0,Garaje no indicado
5,Inmobiliaria E-Castro,venta,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000,NaN,...,sí,5,511,NaN,desconocido,desconocido,otros,0,Dirección sin número,NaN


In [28]:

cols_bool = [
    "ascensor",
    "exterior",
    "garaje_incluido",
]

for col in cols_bool:
    df[col] = (
        df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "desconocido": np.nan,
                "sí": 1,
                "si": 1,
                "no": 0
            })
    )

    # asegurar tipo numérico
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [29]:
df["descuento_flag"] = df["descuento_pct"].notna().astype(int)


In [30]:
df["planta"] = df["planta"].replace({"bajo": 0, "Bajo": 0})
df["planta"] = pd.to_numeric(df["planta"], errors="coerce")


In [31]:
df["tipo_inmueble"] = df["tipo_inmueble"].str.lower().str.strip()
df["subtipo"] = df["subtipo"].str.lower().str.strip()

In [32]:
valores_eliminar = [
    "finca",
    "finca_rustica",
    "garaje",
    "garaje/local",
    "piso/casa",
    "edificio"
]

df = df[~df["tipo_inmueble"].isin(valores_eliminar)]


In [33]:
mask_duplex_atico = df["tipo_inmueble"].isin(["dúplex", "duplex", "ático", "atico"])

df.loc[mask_duplex_atico, "subtipo"] = df.loc[mask_duplex_atico, "tipo_inmueble"]

df.loc[mask_duplex_atico, "tipo_inmueble"] = "piso"


In [34]:
df["tipo_inmueble"] = df["tipo_inmueble"].replace({
    "chalet": "casa"
})


In [35]:
mask_piso_sin_subtipo = (
    (df["tipo_inmueble"] == "piso") &
    (~df["subtipo"].isin(["dúplex", "duplex", "ático", "atico"]))
)

df.loc[mask_piso_sin_subtipo, "subtipo"] = "apartamento"


In [36]:
df["subtipo"].unique()


array(['"avenida riomar', 'barrio brazomar, 46',
       '"calle leonardo rucabado', '"calle de la ronda', 'barrio sámano',
       '"calle de angel perez hornoas cholo"', '"calle de la granja"',
       '"calle ataúlfo argenta"', 'barrio montealegre',
       '"isaac castillo"', '"calle doctor marañón"',
       '"avenida manuel garcía lago"', '"calle leonardo rucabado"',
       'barrio cerdigo', '"avenida maura"', '"avenida de cantabria"',
       'calle del alto de la cruz', '"paseo de pereda"',
       'barrio la arnía', '"avenida de pontejos"',
       '"calle juan de herrera"', '"calle belén', 'barrio sámano, 40',
       'calle aureliano linares rivas',
       'calle aureliano linares rivas, 17', 'calle paco labiano, 5',
       'avenida de chinchapapa', 'calle teresa de calcuta',
       'avenida de francia', 'centro', 'calle padre ignacio ellacuría',
       'el puntal', 'zona playa', 'plaza marqués de albaida',
       'calle peredo y nogales', 'avenida de la libertad, 87',
       'avenid

In [37]:
df["subtipo"] = (
    df["subtipo"]
        .str.lower()
        .str.strip()
        .replace({
            "chalét independiente": "casa independiente",
            "chalét pareado": "casa adosada",
            "chalet pareado": "casa adosada",
            "chalét adosado": "casa adosada",
            "chalet adosado": "casa adosada",
            "casa o chalet independiente": "casa independiente",
            "casa o chalet independent\ufeffe": "casa independiente",
            "chalet": "casa independiente",
            "casa unifamiliar": "casa independiete"
        })
)


In [38]:
df["subtipo"].unique()


array(['"avenida riomar', 'barrio brazomar, 46',
       '"calle leonardo rucabado', '"calle de la ronda', 'barrio sámano',
       '"calle de angel perez hornoas cholo"', '"calle de la granja"',
       '"calle ataúlfo argenta"', 'barrio montealegre',
       '"isaac castillo"', '"calle doctor marañón"',
       '"avenida manuel garcía lago"', '"calle leonardo rucabado"',
       'barrio cerdigo', '"avenida maura"', '"avenida de cantabria"',
       'calle del alto de la cruz', '"paseo de pereda"',
       'barrio la arnía', '"avenida de pontejos"',
       '"calle juan de herrera"', '"calle belén', 'barrio sámano, 40',
       'calle aureliano linares rivas',
       'calle aureliano linares rivas, 17', 'calle paco labiano, 5',
       'avenida de chinchapapa', 'calle teresa de calcuta',
       'avenida de francia', 'centro', 'calle padre ignacio ellacuría',
       'el puntal', 'zona playa', 'plaza marqués de albaida',
       'calle peredo y nogales', 'avenida de la libertad, 87',
       'avenid

In [39]:
df[df["subtipo"].isna()]


,id_anuncio,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,...,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
575,Inmobiliaria JC de Laredo,venta,finca_rustica,barrio rocillo,NaN,Ampuero,Cantabria,320000,NaN,NaN,...,3.0,220,NaN,NaN,NaN,NaN,0,Licencia turística 7 plazas (texto),NaN,0
577,Inmobiliaria Mi Villa,venta,finca_rustica,n-629a,NaN,Rasines,Cantabria,89000,NaN,NaN,...,1.0,300,NaN,NaN,NaN,NaN,0,Casa piedra; parcela 6000 m2 (texto),NaN,0
584,Inmobiliaria Mi Villa,venta,finca_rustica,camino de solorga,NaN,Bareyo,Cantabria,300000,NaN,NaN,...,3.0,200,NaN,NaN,NaN,NaN,0,Dos casas; 38590 m2 (texto),NaN,0


In [40]:
df["tipo_inmueble"] = df["tipo_inmueble"].fillna("casa independiente")


In [41]:
df["direccion_texto"] = (
    df["direccion_texto"]
        .astype(str)
        .str.strip()                  # elimina espacios inicio y final
        .str.strip('"')               # elimina comillas dobles inicio y final
        .str.strip("'")               # elimina comillas simples si hubiera
        .str.strip()                  # limpia otra vez por seguridad
)


In [42]:
df.head(20)


,id_anuncio,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,...,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
1,Inmobiliaria E-Castro,venta,piso,casa independiente,"""avenida riomar",1,Cotolino,Castro-Urdiales,Cantabria,498000,...,1.0,4,121,4.0,1.0,1.0,otros,0,NaN,0
2,Inmobiliarias.com,venta,casa,chalét independiente,"barrio brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000,NaN,...,5.0,232,NaN,NaN,NaN,NaN,0,NaN,NaN,1
3,Inmobiliaria E-Castro,venta,piso,casa independiente,"""calle leonardo rucabado",6,Centro,Castro-Urdiales,Cantabria,299000,...,NaN,2,84,3.0,1.0,1.0,otros,0,Garaje no indicado,0
4,Inmobiliaria E-Castro,venta,piso,casa independiente,"""calle de la ronda",32,Centro,Castro-Urdiales,Cantabria,340000,...,NaN,4,99,5.0,1.0,1.0,otros,0,Garaje no indicado,0
5,Inmobiliaria E-Castro,venta,casa,chalét independiente,barrio sámano,Sámano,Castro-Urdiales,Cantabria,799000,NaN,...,5.0,511,NaN,NaN,NaN,NaN,0,Dirección sin número,NaN,1
6,Inmobiliarias.com,venta,ático,casa independiente,"""calle de angel perez hornoas cholo""",Brazomar,Castro-Urdiales,Cantabria,360000,NaN,...,3.0,130,6,NaN,1.0,NaN,0,NaN,NaN,1
7,Inmobiliarias.com,venta,ático,casa independiente,"""calle de la granja""",Cotolino,Castro-Urdiales,Cantabria,405000,NaN,...,2.0,90,4,NaN,1.0,NaN,0,NaN,NaN,1
8,Inmobiliarias.com,venta,piso,casa independiente,"""calle ataúlfo argenta""",Cotolino,Castro-Urdiales,Cantabria,415000,NaN,...,3.0,110,2,NaN,1.0,NaN,0,NaN,NaN,1
9,Inmobiliarias.com,venta,casa,chalét independiente,barrio montealegre,Sámano,Castro-Urdiales,Cantabria,455000,NaN,...,5.0,266,NaN,NaN,NaN,NaN,0,Garaje no indicado,NaN,1
10,Inmobiliaria E-Castro,venta,piso,casa independiente,"""isaac castillo""",Mioño-Santullán,Castro-Urdiales,Cantabria,180000,NaN,...,4.0,105,1,NaN,0.0,NaN,0,NaN,NaN,1


In [43]:
cols_numericas = [
    "precio_eur",
    "precio_anterior_eur",
    "superficie_m2",
    "habitaciones",
    "planta",
    "descuento_pct"
]

for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in cols_numericas:
    print(col, df[col].isna().sum())




precio_eur 789
precio_anterior_eur 796
superficie_m2 176
habitaciones 100
planta 518
descuento_pct 861


In [44]:
df = df[df["precio_eur"].notna()]


In [45]:
df["precio_eur"].isna().sum()


0

In [46]:
df[cols_numericas].dtypes


precio_eur             float64
precio_anterior_eur    float64
superficie_m2          float64
habitaciones           float64
planta                 float64
descuento_pct          float64
dtype: object

In [47]:
df.dtypes


id_anuncio               object
fuente                   object
operacion                object
tipo_inmueble            object
subtipo                  object
direccion_texto          object
barrio_zona              object
municipio                object
provincia                object
precio_eur              float64
precio_anterior_eur     float64
descuento_pct           float64
garaje_incluido         float64
habitaciones            float64
superficie_m2           float64
planta                  float64
exterior                float64
ascensor                float64
obra_nueva_categoria     object
obra_nueva_flag          object
notas_parseo             object
descuento_flag            int32
dtype: object

In [48]:
df.head(20)

,id_anuncio,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,...,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
1,Inmobiliaria E-Castro,venta,piso,casa independiente,"""avenida riomar",1,Cotolino,Castro-Urdiales,Cantabria,498000.0,...,1.0,4.0,121.0,4.0,1.0,1.0,otros,0,NaN,0
3,Inmobiliaria E-Castro,venta,piso,casa independiente,"""calle leonardo rucabado",6,Centro,Castro-Urdiales,Cantabria,299000.0,...,NaN,2.0,84.0,3.0,1.0,1.0,otros,0,Garaje no indicado,0
4,Inmobiliaria E-Castro,venta,piso,casa independiente,"""calle de la ronda",32,Centro,Castro-Urdiales,Cantabria,340000.0,...,NaN,4.0,99.0,5.0,1.0,1.0,otros,0,Garaje no indicado,0
16,Exclusivas Inmobiliarias Mikeli,venta,piso,casa independiente,"""avenida de cantabria""",Valdenoja,Santander,Cantabria,798000,860000.0,...,3.0,178.0,3.0,NaN,1.0,NaN,0,En el texto indica edificio construido en 2005,NaN,1
23,Inmobiliarias.com,venta,piso,casa independiente,"""calle belén",3,Centro,Castro-Urdiales,Cantabria,430000.0,...,NaN,2.0,84.0,2.0,1.0,1.0,otros,0,Indica 'reciente construcción' pero no 'obra n...,0
24,Inmobiliaria E-Castro,venta,piso,casa independiente,"""calle leonardo rucabado",50,Brazomar,Castro-Urdiales,Cantabria,270000.0,...,1.0,2.0,64.0,2.0,1.0,1.0,otros,0,NaN,0
29,Inmobiliarias.com,venta,piso,casa independiente,"""avenida riomar",1,Cotolino,Castro-Urdiales,Cantabria,498000.0,...,1.0,4.0,120.0,4.0,1.0,1.0,otros,0,Duplicado potencial del anuncio #1 (misma dire...,0
35,Inmobiliaria JC de Laredo,venta,piso,casa independiente,centro,Centro,Laredo,Cantabria,580000,620000.0,...,NaN,4.0,240.0,5.0,1.0,1.0,otros,0,Texto lo describe como ático dúplex,1
41,Trece Home,venta,piso,casa independiente,calle peredo y nogales,Tregadín,Noja,Cantabria,269000,279000.0,...,NaN,2.0,75.0,0.0,NaN,1.0,otros,0,NaN,1
59,Inmobiliaria Uranga,venta,casa,chalét pareado,avenida de la libertad,Zona Playa,Laredo,Cantabria,385000,398000.0,...,NaN,4.0,180.0,NaN,NaN,NaN,otros,0,Zona Puntal según texto; cabecera pone Zona Playa,1
